# Part 5 — PyCaret Capabilities Landscape

This notebook gives a concise, runnable overview of the five main PyCaret capabilities:

1. Classification
2. Regression
3. Clustering
4. Anomaly Detection
5. Time-Series Forecasting

The common pattern is:

**setup → create/compare model → predict/assign results**

## Important Colab setup

This notebook uses **PyCaret 3.3.2**, so in Google Colab select:

**Runtime → Change runtime type → Runtime Version → 2025.07**

That runtime uses Python 3.11.

On the **first run only**, execute the installation cell below, then choose:

**Runtime → Restart session**

After the restart, run the notebook from the top again.


## 1. Install PyCaret 3.3.2

### What we saw before
- This is the starting point of the notebook.
- PyCaret 3.3.2 needs a Python 3.11 environment and older compatible scientific-package versions.

### How this cell is different
- This cell only prepares the software environment.
- It does not train any machine-learning model.

### What this cell does
- Verifies that Python 3.11 is active.
- Installs PyCaret 3.3.2 with the optional analysis/model dependencies used by the PyCaret 3.x ecosystem.

### What we achieve after running it
- The required PyCaret version and compatible dependencies are installed.
- After the first installation, restart the Colab session once and rerun from the top.

### Remember
Use:

**Colab Runtime Version 2025.07 → Python 3.11 → PyCaret 3.3.2**


In [ ]:
import sys

print("Python version:", sys.version.split()[0])

assert sys.version_info[:2] == (3, 11), (
    "This notebook needs Python 3.11. "
    "In Colab choose Runtime -> Change runtime type -> "
    "Runtime Version -> 2025.07, then reconnect."
)

!pip install -q --only-binary=:all: "pycaret[analysis,models]==3.3.2"

print("PyCaret installation finished.")
print("FIRST RUN ONLY: Runtime -> Restart session, then run the notebook from the top.")


## 2. Import the Required Libraries

### What we saw before
- PyCaret 3.3.2 has been installed.
- After the one-time runtime restart, we can safely import the compatible package versions.

### How this cell is different
- This cell loads the five PyCaret Experiment classes used in the rest of the notebook.

### What this cell does
- Imports NumPy, Pandas, and scikit-learn datasets.
- Imports:
  - `ClassificationExperiment`
  - `RegressionExperiment`
  - `ClusteringExperiment`
  - `AnomalyExperiment`
  - `TSForecastingExperiment`
- Prints key package versions for verification.
- Sets a reproducible random seed.

### What we achieve after running it
- All five PyCaret modules are ready.
- The printed versions make the environment easy to verify.

### Remember
The five Experiment classes correspond to five different ML problem types, but their workflow is intentionally similar.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd
import sklearn
import pycaret

from IPython.display import display
from sklearn.datasets import load_breast_cancer, load_diabetes, make_blobs

# Import LightGBM before sklearn-backed PyCaret training, matching the stable 3.x setup.
import lightgbm

from pycaret.classification import ClassificationExperiment
from pycaret.regression import RegressionExperiment
from pycaret.clustering import ClusteringExperiment
from pycaret.anomaly import AnomalyExperiment
from pycaret.time_series import TSForecastingExperiment

RANDOM_STATE = 42
N_JOBS = 1
rng = np.random.default_rng(RANDOM_STATE)

print("PyCaret:", pycaret.__version__)
print("Python:", os.sys.version.split()[0])
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)

assert pycaret.__version__ == "3.3.2"
assert int(np.__version__.split(".")[0]) == 1, "Restart the runtime after installation."
assert tuple(map(int, pd.__version__.split(".")[:2])) < (2, 2), "Restart the runtime after installation."

print("Environment verification passed.")


## 3. Classification — Predict a Category

### What we saw before
- The PyCaret environment is ready.
- We can now start with a supervised-learning task.

### How this cell is different
- Classification predicts a **category**.
- Here the target has two possible classes.

### What this cell does
- Loads the breast-cancer dataset.
- Creates a `ClassificationExperiment`.
- `setup()` defines the data, target, folds, seed, and CPU usage.
- `compare_models()` compares Logistic Regression and Random Forest.
- `pull()` retrieves the model-comparison table.
- `predict_model()` generates predictions on PyCaret's held-out partition.

### What we achieve after running it
- We get a small classification leaderboard.
- We obtain predictions from the best classifier.

### Remember
Classification asks:

**Which class does this row belong to?**

Core flow:

```python
setup → compare_models → pull → predict_model
```


In [ ]:
cancer = load_breast_cancer(as_frame=True)
clf_df = cancer.frame.copy()

clf_exp = ClassificationExperiment()

clf_exp.setup(
    data=clf_df,
    target="target",
    session_id=RANDOM_STATE,
    fold=2,
    n_jobs=N_JOBS,
    verbose=False,
    html=False
)

clf_best = clf_exp.compare_models(
    include=["lr", "rf"],
    sort="AUC",
    fold=2,
    verbose=False
)

clf_leaderboard = clf_exp.pull()

print("Classification leaderboard:")
display(clf_leaderboard)

clf_predictions = clf_exp.predict_model(
    clf_best,
    verbose=False
)

print("Sample predictions:")
display(
    clf_predictions[
        ["target", "prediction_label", "prediction_score"]
    ].head()
)


## 4. Regression — Predict a Number

### What we saw before
- Classification predicted a category.
- We used `setup()`, `compare_models()`, and `predict_model()`.

### How this cell is different
- Regression predicts a **continuous numerical value** instead of a class.
- The workflow remains almost the same.

### What this cell does
- Loads the diabetes regression dataset.
- Creates a `RegressionExperiment`.
- Compares Linear Regression and Random Forest.
- Sorts the models using MAE.
- Generates numerical predictions on the holdout partition.

### What we achieve after running it
- We see the same PyCaret workflow applied to a regression problem.
- We get a leaderboard and predicted target values.

### Remember
Classification predicts:

**a category**

Regression predicts:

**a number**

For MAE:

**smaller is better.**


In [ ]:
diabetes = load_diabetes(as_frame=True)
reg_df = diabetes.frame.copy()

reg_exp = RegressionExperiment()

reg_exp.setup(
    data=reg_df,
    target="target",
    session_id=RANDOM_STATE,
    fold=2,
    n_jobs=N_JOBS,
    verbose=False,
    html=False
)

reg_best = reg_exp.compare_models(
    include=["lr", "rf"],
    sort="MAE",
    fold=2,
    verbose=False
)

reg_leaderboard = reg_exp.pull()

print("Regression leaderboard:")
display(reg_leaderboard)

reg_predictions = reg_exp.predict_model(
    reg_best,
    verbose=False
)

print("Sample predictions:")
display(
    reg_predictions[
        ["target", "prediction_label"]
    ].head()
)


## 5. Clustering — Discover Groups

### What we saw before
- Classification and regression were supervised tasks.
- Both had a known target column.

### How this cell is different
- Clustering is **unsupervised**.
- There is no target column.

### What this cell does
- Creates a small synthetic dataset containing four natural groups.
- Creates a `ClusteringExperiment`.
- Normalizes the features.
- `create_model("kmeans")` builds a four-cluster K-Means model.
- `assign_model()` attaches the discovered cluster label to every row.

### What we achieve after running it
- Every observation receives a cluster assignment.
- We can inspect the discovered cluster sizes.

### Remember
Supervised learning:

**features + target**

Clustering:

**features only → discover groups**

Core unsupervised flow:

```python
setup → create_model → assign_model
```


In [ ]:
X_cluster, _ = make_blobs(
    n_samples=250,
    centers=4,
    n_features=3,
    cluster_std=1.0,
    random_state=RANDOM_STATE
)

cluster_df = pd.DataFrame(
    X_cluster,
    columns=["feature_1", "feature_2", "feature_3"]
)

cluster_exp = ClusteringExperiment()

cluster_exp.setup(
    data=cluster_df,
    normalize=True,
    session_id=RANDOM_STATE,
    n_jobs=N_JOBS,
    verbose=False,
    html=False
)

kmeans_model = cluster_exp.create_model(
    "kmeans",
    num_clusters=4,
    verbose=False
)

clustered = cluster_exp.assign_model(
    kmeans_model,
    verbose=False
)

display(clustered.head())

print("Cluster sizes:")
print(clustered["Cluster"].value_counts().sort_index())


## 6. Anomaly Detection — Find Unusual Rows

### What we saw before
- Clustering grouped similar rows together.
- It did not explicitly identify abnormal observations.

### How this cell is different
- Anomaly detection focuses on identifying observations that are unusual compared with the majority.
- It is also an unsupervised task.

### What this cell does
- Creates mostly normal observations and adds a small group of extreme outliers.
- Creates an `AnomalyExperiment`.
- Trains an Isolation Forest using `create_model("iforest")`.
- `fraction=0.05` tells the model that approximately 5% of observations are expected to be anomalous.
- `assign_model()` attaches anomaly labels and scores.

### What we achieve after running it
- Rows with `Anomaly = 1` are flagged as unusual.
- `Anomaly_Score` helps rank the unusual observations.

### Remember
Clustering asks:

**Which group?**

Anomaly detection asks:

**Is this row unusual?**


In [ ]:
normal = rng.normal(
    loc=0,
    scale=1,
    size=(285, 3)
)

outliers = rng.normal(
    loc=7,
    scale=0.7,
    size=(15, 3)
)

anomaly_df = pd.DataFrame(
    np.vstack([normal, outliers]),
    columns=["x1", "x2", "x3"]
)

anomaly_exp = AnomalyExperiment()

anomaly_exp.setup(
    data=anomaly_df,
    session_id=RANDOM_STATE,
    n_jobs=N_JOBS,
    verbose=False,
    html=False
)

iforest_model = anomaly_exp.create_model(
    "iforest",
    fraction=0.05,
    verbose=False
)

anomaly_scored = anomaly_exp.assign_model(
    iforest_model,
    verbose=False
)

print("Anomaly counts:")
print(anomaly_scored["Anomaly"].value_counts())

print("Highest anomaly scores:")
display(
    anomaly_scored
    .sort_values("Anomaly_Score", ascending=False)
    .head(8)
)


## 7. Time-Series Forecasting — Predict Future Values

### What we saw before
- Classification, regression, clustering, and anomaly detection worked with ordinary feature rows.
- Time-series forecasting is different because observations are ordered in time.

### How this cell is different
- We now use historical values to predict future values.
- The forecast horizon is seven days.
- A single lightweight Naive forecaster is used so the section remains fast.

### What this cell does
- Creates a 90-day daily sales series.
- Gives the data a proper time index.
- Creates a `TSForecastingExperiment`.
- Sets `seasonal_period=7` explicitly, avoiding extra automatic seasonality detection.
- `create_model("naive")` trains a simple baseline forecasting model.
- `predict_model()` generates the seven-day forecast.

### What we achieve after running it
- We get a complete PyCaret time-series example without a heavy model search.
- We can inspect both cross-validation metrics and future predictions.

### Remember
Time series follows:

**past ordered observations → future values**

The most important rule is:

**time order matters.**


In [ ]:
dates = pd.date_range(
    "2026-01-01",
    periods=90,
    freq="D"
)

t = np.arange(len(dates))

sales = (
    100
    + 0.15 * t
    + 12 * np.sin(2 * np.pi * t / 7)
    + rng.normal(0, 4, len(t))
)

sales_series = pd.Series(
    sales,
    index=dates,
    name="sales"
)

ts_exp = TSForecastingExperiment()

ts_exp.setup(
    data=sales_series,
    fh=7,
    fold=2,
    seasonal_period=7,
    ignore_seasonality_test=True,
    session_id=RANDOM_STATE,
    n_jobs=N_JOBS,
    verbose=False,
    html=False
)

naive_model = ts_exp.create_model(
    "naive",
    fold=2,
    verbose=False
)

ts_results = ts_exp.pull()

print("Time-series cross-validation results:")
display(ts_results)

forecast = ts_exp.predict_model(
    naive_model,
    fh=7,
    verbose=False
)

print("7-day forecast:")
display(forecast)


# Final Revision Summary

## PyCaret task map

| Task | Experiment class | Output |
|---|---|---|
| Classification | `ClassificationExperiment` | Class/category |
| Regression | `RegressionExperiment` | Numeric value |
| Clustering | `ClusteringExperiment` | Cluster label |
| Anomaly Detection | `AnomalyExperiment` | Anomaly flag + score |
| Time Series | `TSForecastingExperiment` | Future values |

## Supervised workflow

```python
exp = ClassificationExperiment()

exp.setup(
    data=df,
    target="target"
)

best = exp.compare_models()

predictions = exp.predict_model(best)
```

## Unsupervised workflow

```python
exp = ClusteringExperiment()

exp.setup(data=df)

model = exp.create_model("kmeans")

result = exp.assign_model(model)
```

## Main takeaway

PyCaret provides a common low-code workflow across different machine-learning tasks:

**prepare data → configure experiment → build/compare model → inspect results**

The task changes, but the overall workflow remains consistent.
